# 03 — Decorators

A **decorator** is a function that takes a function and returns a (usually wrapped) function. The `@` syntax is just sugar for `f = decorator(f)`.

Once you've seen the desugaring once, every decorator in FastAPI / Pydantic / agent frameworks will read the same way.

## The desugaring

```python
@my_decorator
def greet(name):
    return f"Hello, {name}"
```

is **exactly** equivalent to:

```python
def greet(name):
    return f"Hello, {name}"
greet = my_decorator(greet)
```

So a decorator is just a function that takes a function and returns something. Usually it returns a *new function* that calls the original, with extra behavior wrapped around it.

## A decorator that does nothing useful (yet)

In [1]:
def announce(func):
    # `func` is the original function we're decorating
    def wrapper(*args, **kwargs):
        print(f"-> calling {func.__name__}")
        result = func(*args, **kwargs)       # call the original
        print(f"<- {func.__name__} returned {result!r}")
        return result
    return wrapper                            # return the wrapped function

@announce
def add(a, b):
    return a + b

print(add(2, 3))
print(add(10, 20))

-> calling add
<- add returned 5
5
-> calling add
<- add returned 30
30


Notice how `wrapper(*args, **kwargs)` lets the decorator handle **any signature** — that's why `*args`/`**kwargs` from the previous notebook matters here.

## A useful decorator: `@timed`

In [2]:
import time

def timed(func):
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{func.__name__} took {elapsed * 1000:.2f} ms")
    return wrapper

@timed
def slow_sum(n):
    return sum(range(n))

print(slow_sum(1_000_000))
print(slow_sum(5_000_000))

slow_sum took 17.47 ms
499999500000
slow_sum took 81.13 ms
12499997500000


## The metadata problem — use `functools.wraps`

The wrapped function above has lost its identity:

In [3]:
print(slow_sum.__name__)   # 'wrapper' — wrong!
print(slow_sum.__doc__)    # gone
help(slow_sum)             # documentation tools see 'wrapper'

wrapper
None
Help on function wrapper in module __main__:

wrapper(*args, **kwargs)



`functools.wraps` copies metadata (`__name__`, `__doc__`, `__module__`, etc.) from the wrapped function onto the wrapper. **Always use it.**

In [4]:
import time
import functools

def timed(func):
    @functools.wraps(func)        # <-- the fix
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        try:
            return func(*args, **kwargs)
        finally:
            elapsed = time.perf_counter() - start
            print(f"{func.__name__} took {elapsed * 1000:.2f} ms")
    return wrapper

@timed
def slow_sum(n):
    """Sum the integers 0..n-1."""
    return sum(range(n))

print(slow_sum.__name__)   # 'slow_sum' — fixed
print(slow_sum.__doc__)    # docstring preserved
print(slow_sum(1_000_000))

slow_sum
Sum the integers 0..n-1.
slow_sum took 15.62 ms
499999500000


## Decorators with arguments — one more layer

Plain decorator: `decorator(func) -> wrapper`.

Decorator with arguments: `decorator_factory(args) -> decorator(func) -> wrapper`. Three levels deep — slow down.

In [5]:
import functools
import time

def retry(times=3, delay=0.1):
    # Outer call: retry(times=3) — returns the actual decorator.
    def decorator(func):
        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            last_exc = None
            for attempt in range(1, times + 1):
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    last_exc = e
                    print(f"attempt {attempt}/{times} failed: {e}")
                    time.sleep(delay)
            raise last_exc
        return wrapper
    return decorator

@retry(times=3, delay=0.05)
def flaky():
    import random
    if random.random() < 0.7:
        raise RuntimeError("transient")
    return "ok"

import random; random.seed(0)
print(flaky())

ok


## Stacking decorators

Decorators apply **bottom-up** — the one closest to `def` runs first.

In [6]:
@announce        # outer: applied second
@timed           # inner: applied first
def add(a, b):
    return a + b

# Equivalent to:  add = announce(timed(add))
add(2, 3)

-> calling add
add took 0.00 ms
<- add returned 5


5

## Built-in decorators you'll meet soon

- `@functools.lru_cache(maxsize=...)` — memoize results.
- `@classmethod`, `@staticmethod`, `@property` — covered in the OOP folders.
- `@dataclass` — covered in folder 04.
- `@app.get("/users/{id}")` — FastAPI, folder 13.

In [7]:
import functools

@functools.lru_cache(maxsize=128)
def fib(n):
    if n < 2:
        return n
    return fib(n - 1) + fib(n - 2)

print(fib(100))
print(fib.cache_info())

354224848179261915075
CacheInfo(hits=98, misses=101, maxsize=128, currsize=101)


## Mini-exercise

1. Write `@log_calls` that prints `func.__name__`, the args/kwargs, and the return value. Preserve metadata with `wraps`.
2. Write `@require_positive` that raises `ValueError` if any positional argument is < 0. Apply to a function and verify both paths.
3. Convert this code to use a decorator:
   ```python
   def fetch():
       for attempt in range(3):
           try:
               return do_request()
           except Exception:
               pass
       raise RuntimeError("gave up")
   ```

In [8]:
# your solutions here


**Takeaways**

- `@d` over `def f` is just `f = d(f)`.
- A decorator usually returns `wrapper(*args, **kwargs)` that calls the original.
- **Always** use `@functools.wraps(func)` on the wrapper.
- A decorator *with arguments* is a function that returns a decorator — three levels of nesting.
- Stacked decorators apply **bottom-up**.

Next: [04_decorator_module/](04_decorator_module/) — a real, importable decorator module you'd actually keep.